In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load dataset
df = pd.read_csv('datafinal.csv')
df.head()

,IDJwb,IDPSJ,questions,answerKeys,answer,raw_grade,max_grade,grade,labela,label
0,1,1,Ada empat macam karakter atau olah yang diusul...,Olah pikir = Berpikiran kritis dalam segala si...,Olah pikir = Berpikiran kritis dalam segala si...,50.0,50.0,10.0,A,5
1,2,1,Ada empat macam karakter atau olah yang diusul...,Olah pikir = Berpikiran kritis dalam segala si...,KARAKTER SALING MENOLONG\nKARAKTER YANG MAU SA...,30.0,50.0,6.0,C,3
2,3,1,Ada empat macam karakter atau olah yang diusul...,Olah pikir = Berpikiran kritis dalam segala si...,1. olah hati adalah karakter yang penuh rasa d...,50.0,50.0,10.0,A,5
3,4,1,Ada empat macam karakter atau olah yang diusul...,Olah pikir = Berpikiran kritis dalam segala si...,Revolusi mental\nyang berarti kita harus memil...,20.0,50.0,4.0,D,2
4,5,1,Ada empat macam karakter atau olah yang diusul...,Olah pikir = Berpikiran kritis dalam segala si...,1. Olah raga : memiliki karakter yang kuat\n2....,50.0,50.0,10.0,A,5


In [3]:
# Hapus kolom yang tidak dipakai (hanya pakai sampai kolom label)
kolom_yang_dipakai = ['IDJwb', 'IDPSJ', 'questions', 'answerKeys', 'answer', 
                        'raw_grade', 'max_grade', 'grade', 'labela', 'label']
df = df[kolom_yang_dipakai]

In [4]:
# Preprocessing Step 1: Lowercase and single space
import re
def lowercase(text):
    """
    Tahap 1: Lowercase dan single space
    - Ubah semua text menjadi lowercase
    - Hapus extra spaces (multiple spaces menjadi single space)
    - Pertahankan newline untuk pemisahan kalimat
    - Menangani non-breaking space (\xa0) dan whitespace lainnya
    """
    if pd.isna(text):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Ganti semua whitespace (termasuk \xa0, \t, dll) KECUALI newline dengan spasi biasa
    # \xa0 adalah non-breaking space yang sering muncul dari copy-paste
    text = text.replace('\xa0', ' ')  # Non-breaking space
    text = text.replace('\t', ' ')     # Tab
    text = text.replace('\r', '')      # Carriage return
    
    # Normalize multiple spaces menjadi single space (TIDAK termasuk newline)
    text = re.sub(r'[ ]+', ' ', text)
    
    # Normalize multiple newlines menjadi single newline
    text = re.sub(r'\n+', '\n', text)
    
    # Hapus spasi di awal/akhir setiap baris
    lines = text.split('\n')
    lines = [line.strip() for line in lines]
    text = '\n'.join(lines)
    
    return text

In [5]:
# Preprocessing Step 2: Sentence Tokenization menggunakan Regex
import re

def sentence_tokenize(text):
    """
    Tahap 2: Sentence Tokenization menggunakan regex
    Pemisah kalimat:
    1. Newline (\n)
    2. Nomor/huruf penanda di tengah dengan tanda baca: "1.", "1)", "a.", "a)"
       (hanya jika diikuti spasi + teks, agar angka/huruf di akhir kalimat tidak terpotong)
    3. Titik koma ";"
    4. Titik "." diikuti spasi
    Setelah split, penanda di awal kalimat dihapus: nomor, huruf, dash, *, .
    """
    if not text or pd.isna(text):
        return []

    # Pisahkan dulu berdasarkan newline
    lines = text.split('\n')

    sentences = []
    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Pisahkan berdasarkan pola penomoran di tengah baris
        # Hanya menangani "1.", "1)", "a.", "a)" — TIDAK "1 " (terlalu ambigu)
        # \s+\S memastikan ada teks setelah penanda,
        # sehingga angka/huruf di akhir kalimat (misal "ada 4." atau "partisi d.") tidak terpotong
        sub_parts = re.split(r'\s+(?=[0-9]{1,2}[\.\)]\s+\S|[a-zA-Z][\.\)]\s+\S)', line)

        for part in sub_parts:
            # Pisahkan berdasarkan ";" (titik koma)
            semi_parts = re.split(r'\s*;\s*', part)
            for sp in semi_parts:
                # Pisahkan berdasarkan ". " (titik diikuti spasi)
                dot_parts = re.split(r'\.\s+', sp)
                for dp2 in dot_parts:
                    dp2 = dp2.strip()
                    # Strip tanda kutip di awal/akhir terlebih dahulu
                    # agar penanda angka/huruf di awal bisa terdeteksi
                    dp2 = re.sub(r'^["""\'\'\']+|["""\'\'\']+$', '', dp2).strip()
                    # Hapus penanda di awal kalimat:
                    # "1.", "1)", "a.", "a)"  → penomoran eksplisit
                    # "1 "                   → angka diikuti spasi di awal
                    # "1"                    → angka berdiri sendiri (sisa split titik)
                    # "a"                    → huruf berdiri sendiri (sisa split titik)
                    # "- ", "* ", ". "       → simbol bullet di awal
                    dp2 = re.sub(
                        r'^[0-9]{1,2}[\.\)]\s*'   # 1. atau 1)
                        r'|^[0-9]{1,2}\s+'         # 1 (angka + spasi)
                        r'|^[0-9]{1,2}$'           # 1 (angka berdiri sendiri)
                        r'|^[a-zA-Z][\.\)]\s*'     # a. atau a)
                        r'|^[a-zA-Z]$'             # a (huruf berdiri sendiri)
                        r'|^-\s*'                  # - (dash, dengan atau tanpa spasi)
                        r'|^\*\s*'                 # * (asterisk/bullet)
                        r'|^\.\s+',                # . (titik di awal)
                        '', dp2
                    ).strip()
                    # Hapus titik di akhir kalimat
                    dp2 = dp2.rstrip('.')
                    if dp2:
                        sentences.append(dp2)

    return sentences


In [6]:
# Preprocessing Step 3: Normalisasi Simbol
def normalize_symbols(sentences):
    """
    Tahap 3: Normalisasi simbol khusus
    - : dan = menjadi "adalah"
    - / menjadi "atau"
    - ->, -->, --->, dst. (satu atau lebih dash diikuti >) menjadi "kemudian"
    - > menjadi "lebih besar"
    - < menjadi "lebih kecil"
    - & menjadi "dan"
    URL/link dipertahankan utuh.
    """
    if not sentences:
        return []
    
    normalized_sentences = []
    for sentence in sentences:
        # Simpan URL sebagai placeholder tanpa tanda baca agar tidak rusak saat normalisasi simbol
        urls = re.findall(r'https?://\S+|www\.\S+', sentence)
        for idx, url in enumerate(urls):
            sentence = sentence.replace(url, f'URLTOKEN{idx}')

        # Ganti ->, -->, --->, dst. dengan "kemudian" (harus sebelum > diganti)
        sentence = re.sub(r'-+>', ' kemudian ', sentence)
        
        # Ganti : dan = dengan "adalah"
        sentence = sentence.replace(':', ' adalah ')
        sentence = sentence.replace('=', ' adalah ')
        
        # Ganti / dengan "atau"
        sentence = sentence.replace('/', ' atau ')

        # Ganti > dengan "lebih besar"
        sentence = sentence.replace('>', ' lebih besar dari ')

        # Ganti < dengan "lebih kecil"
        sentence = sentence.replace('<', ' lebih kecil dari ')

        # Ganti & dengan "dan"
        sentence = sentence.replace('&', ' dan ')

        # Kembalikan URL
        for idx, url in enumerate(urls):
            sentence = sentence.replace(f'URLTOKEN{idx}', url)
        
        # Hapus extra spaces yang mungkin muncul
        sentence = re.sub(r' +', ' ', sentence).strip()
        
        if sentence:  # Hanya tambahkan jika tidak kosong
            normalized_sentences.append(sentence)
    
    return normalized_sentences


In [7]:
# Preprocessing Step 4: Menghilangkan Tanda Baca
import string

def remove_punctuation(sentences):
    """
    Tahap 4: Menghilangkan tanda baca !"#$%&'()*+,-./:;<=>?@[]^_{|}~`
    Menghapus semua tanda baca dari kalimat-kalimat,
    kecuali URL/link yang dipertahankan utuh.
    """
    if not sentences:
        return []
    
    cleaned_sentences = []
    for sentence in sentences:
        # Simpan URL sebagai placeholder tanpa tanda baca agar tidak rusak saat translate
        urls = re.findall(r'https?://\S+|www\.\S+', sentence)
        for idx, url in enumerate(urls):
            sentence = sentence.replace(url, f'URLTOKEN{idx}')

        # Hapus semua tanda baca
        translator = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
        clean_sentence = sentence.translate(translator)

        # Kembalikan URL
        for idx, url in enumerate(urls):
            clean_sentence = clean_sentence.replace(f'URLTOKEN{idx}', url)

        # Hapus extra spaces yang mungkin muncul
        clean_sentence = re.sub(r' +', ' ', clean_sentence).strip()
        
        if clean_sentence:  # Hanya tambahkan jika tidak kosong
            cleaned_sentences.append(clean_sentence)
    
    return cleaned_sentences


In [8]:
df['questions_lower'] = df['questions'].apply(lowercase)
df['answerKeys_lower'] = df['answerKeys'].apply(lowercase)
df['answer_lower'] = df['answer'].apply(lowercase)

In [9]:
# Terapkan pemotongan kalimat
df['answerKeys_sentences'] = df['answerKeys_lower'].apply(sentence_tokenize)
df['questions_sentences'] = df['questions_lower'].apply(sentence_tokenize)
df['answer_sentences'] = df['answer_lower'].apply(sentence_tokenize)

In [10]:
# Terapkan normalisasi simbol
df['answerKeys_no_punct'] = df['answerKeys_sentences'].apply(normalize_symbols)
df['questions_no_punct'] = df['questions_sentences'].apply(normalize_symbols)
df['answer_no_punct'] = df['answer_sentences'].apply(normalize_symbols)

In [11]:
# Terapkan penghapusan tanda baca
df['answerKeys_clean'] = df['answerKeys_no_punct'].apply(remove_punctuation)
df['questions_clean'] = df['questions_no_punct'].apply(remove_punctuation)
df['answer_clean'] = df['answer_no_punct'].apply(remove_punctuation)

In [12]:
for i in range(len(df)):
    print(f"Data ke-{i}:")
    print('Sebelum preprocessing:')
    print(df['answer'][i])
    print('Sesudah preprocessing:')
    print(df['answer_clean'][i])
    print()

Data ke-0:
Sebelum preprocessing:
Olah pikir = Berpikiran kritis dalam segala situasi yang terjadi.

Olah hati = Melakukan suatu kegiatan berdasarkan hati nurani kita.

Olah raga = Memiliki sifat dan mental yang kuat.

Olah hati dan karsa = Kreatif dalam melakukan sesuatu.
Sesudah preprocessing:
['olah pikir adalah berpikiran kritis dalam segala situasi yang terjadi', 'olah hati adalah melakukan suatu kegiatan berdasarkan hati nurani kita', 'olah raga adalah memiliki sifat dan mental yang kuat', 'olah hati dan karsa adalah kreatif dalam melakukan sesuatu']

Data ke-1:
Sebelum preprocessing:
KARAKTER SALING MENOLONG
KARAKTER YANG MAU SALING BEKERJA SAMA
KARAKTER YANG PEDULI TERHADAP PENDERITAAN ORANG LAIN
KARAKTER YANG MEMENTINGKAN SIKAP RELA BERKORBAN
Sesudah preprocessing:
['karakter saling menolong', 'karakter yang mau saling bekerja sama', 'karakter yang peduli terhadap penderitaan orang lain', 'karakter yang mementingkan sikap rela berkorban']

Data ke-2:
Sebelum preprocessing:
1. 

In [ ]:
# # Simpan hasil preprocessing ke TXT untuk perbandingan
# with open("hasil_regex.txt", "w", encoding="utf-8") as f:
#     for i, row in enumerate(df['answer_clean']):
#         f.write(f"Data {i + 1}:\n")
#         if isinstance(row, list):
#             for sentence in row:
#                 f.write(f"  {sentence}\n")
#         else:
#             f.write(f"  {row}\n")
#         f.write("\n")

# print("✓ Hasil preprocessing disimpan ke: hasil_regex.txt")


✓ Hasil preprocessing disimpan ke: hasil_regex.txt


In [13]:
df = df[['IDJwb', 'IDPSJ', 'questions_clean', 'answerKeys_clean', 'answer_clean', 
                        'raw_grade', 'max_grade', 'grade', 'labela', 'label',]]

df.head()

,IDJwb,IDPSJ,questions_clean,answerKeys_clean,answer_clean,raw_grade,max_grade,grade,labela,label
0,1,1,[ada empat macam karakter atau olah yang diusu...,[olah pikir adalah berpikiran kritis dalam seg...,[olah pikir adalah berpikiran kritis dalam seg...,50.0,50.0,10.0,A,5
1,2,1,[ada empat macam karakter atau olah yang diusu...,[olah pikir adalah berpikiran kritis dalam seg...,"[karakter saling menolong, karakter yang mau s...",30.0,50.0,6.0,C,3
2,3,1,[ada empat macam karakter atau olah yang diusu...,[olah pikir adalah berpikiran kritis dalam seg...,[olah hati adalah karakter yang penuh rasa dam...,50.0,50.0,10.0,A,5
3,4,1,[ada empat macam karakter atau olah yang diusu...,[olah pikir adalah berpikiran kritis dalam seg...,"[revolusi mental, yang berarti kita harus memi...",20.0,50.0,4.0,D,2
4,5,1,[ada empat macam karakter atau olah yang diusu...,[olah pikir adalah berpikiran kritis dalam seg...,"[olah raga adalah memiliki karakter yang kuat,...",50.0,50.0,10.0,A,5


In [14]:
# Simpan hasil preprocessing ke CSV
output_filename = 'datafinal_preprocessed.csv'

# Simpan ke CSV
df.to_csv(output_filename, index=False, encoding='utf-8')

print(f"✓ Data berhasil disimpan ke: {output_filename}")
print(f"  - Total baris: {len(df)}")
print(f"  - Total kolom: {len(df.columns)}")
print(f"  - Kolom: {list(df.columns)}")

✓ Data berhasil disimpan ke: datafinal_preprocessed.csv
  - Total baris: 1229
  - Total kolom: 10
  - Kolom: ['IDJwb', 'IDPSJ', 'questions_clean', 'answerKeys_clean', 'answer_clean', 'raw_grade', 'max_grade', 'grade', 'labela', 'label']
